# Part 1 - Multimodal RAG
DATA 266 Lab 2, Pair_Programming_Team_32. The code lives in `part1_rag/src/`; this notebook runs it.
Run on Colab (CPU is enough): Runtime → Restart & Run All.

In [1]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
except ImportError:
    gpu = "cpu"
print({"commit": commit, "python": sys.version.split()[0], "gpu": gpu, "seed": SEED})

{'commit': 'b1eff3c', 'python': '3.11.17', 'gpu': 'cpu', 'seed': 266}


## Setup
Colab secrets needed: `ANTHROPIC_API_KEY` (answers) and `GEMINI_API_KEY`. When asked, upload `repo.bundle` (made with `git bundle create repo.bundle zoheb`).

In [2]:
import os
BRANCH = "zoheb"                                            # switch to main once merged
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files                      # no GitHub token: upload repo.bundle (git bundle create repo.bundle zoheb)
        files.upload()
        !git clone -q -b {BRANCH} repo.bundle data266-lab2-team32
    %cd data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai anthropic pydantic
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import anthropic, chromadb, pymupdf, sentence_transformers, google.genai
from part1_rag.src import common
print({"commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "pymupdf": pymupdf.__version__, "chromadb": chromadb.__version__,
       "sentence-transformers": sentence_transformers.__version__, "google-genai": google.genai.__version__, "anthropic": anthropic.__version__,
       "answers": common.CLAUDE_MODEL, "figure transcriptions": common.GEMINI_MODEL})

/Users/zohebw/Desktop/DATA 266/data266-lab2-team32/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 22:29:36.712000 73179 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


{'commit': 'b1eff3c', 'pymupdf': '2.1', 'chromadb': '1.5.9', 'sentence-transformers': '6.2.0', 'google-genai': '3.0.0', 'model': 'gemini-3.5-flash-lite'}


## Data
The competition files are not in git. Upload the 3 PDFs and `Questions.csv` from Kaggle when asked.

In [3]:
need = [d["file"] for d in common.DOCS.values()] + ["Questions.csv"]
missing = [f for f in need if not (common.DATA / f).exists()]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (common.DATA / name.replace(" (1)", "")).write_bytes(data)
missing = [f for f in need if not (common.DATA / f).exists()]
assert not missing, f"missing data files: {missing}"

## 1.1 Ingest and index
Pages become text chunks; every figure and table is cropped. Figure transcriptions come from the committed
cache `outputs/figure_descriptions.json`, so this step makes no API calls.

In [4]:
from part1_rag.src import ingest, index
chunks, figures = ingest.ingest_all()
index.build_index()

DOC1: 37 pages, 17 figures/tables: Table 1-1, Figure 1-1, Figure 1-2, Figure 1-3, Table 1-2, Figure 1-4, Table 1-3, Figure 1-5, Figure 1-6, Table 1-4, Figure 2-1, Figure 2-2, Figure 2-3, Figure 2-4, Figure 2-5, Figure 2-6, Figure 2-7


DOC2: 7 pages, 10 figures/tables: Table A-1, Figure A-1, Table A-2, Figure A-2, Figure A-3, Figure A-4, Figure A-5, Figure A-6, Figure A-7, Table A-3


DOC3: 7 pages, 9 figures/tables: Table B-1, Figure B-1, Figure B-2, Figure B-3, Figure B-4, Table B-2, Figure B-5, Figure B-5 (corrected), Figure C-1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12880.26it/s]

indexed 87 items (36 figures/tables)


Collection(name=knowledge_base)

## 1.2-1.5 Answer Questions.csv
One Claude call per question (40 for the public set). Paid API: roughly $1-2 per full run on Sonnet 5.5.

In [5]:
from part1_rag.src import pipeline
sub, seconds = pipeline.run()
print("median seconds/question:", round(float(np.median(list(seconds.values()))), 2))
print(sub["status"].value_counts().to_dict())
assert pipeline.validate()
sub.head(10)

Q001 answered | NONE | DOC1:p1


Q002 answered | NONE | DOC1:p24


Q003 answered | NONE | DOC1:p17,DOC1:p20,DOC1:p21


Q004 answered | NONE | DOC1:p1


Q005 answered | NONE | DOC1:p3,DOC1:p5,DOC3:p5


Q006 answered | NONE | DOC1:p11


... (32 outputs left out)

Q039 session_override | NONE | DOC2:p3


Q040 session_override | Table A-2 | DOC2:p3,DOC2:Table A-2
median seconds/question: 3.11
{'answered': 33, 'session_override': 4, 'conflict': 1, 'not_found': 1, 'false_premise': 1}


valid: 40 rows



,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,The global economic crisis around 2008 was spa...,NONE,DOC1:p1,answered
1,2,C002,Q002,Economists worry about rising unemployment for...,NONE,DOC1:p24,answered
2,3,C003,Q003,Economists measure economic growth without pri...,NONE,"DOC1:p17,DOC1:p20,DOC1:p21",answered
3,4,C004,Q004,"During the 2009 recession, the world economy s...",NONE,DOC1:p1,answered
4,5,C005,Q005,U.S. unemployment increased dramatically after...,NONE,"DOC1:p3,DOC1:p5,DOC3:p5",answered
5,6,C006,Q006,"Before the crisis, China's output grew at an a...",NONE,DOC1:p11,answered
6,7,C007,Q007,The 2008 crisis tanked stock markets worldwide...,"Figure 1-1,Figure A-4","DOC1:p2,DOC2:p5,DOC1:Figure 1-1,DOC2:Figure A-4",answered
7,8,C008,Q008,Not necessarily. Although Okun's law shows tha...,NONE,DOC1:p29,answered
8,9,C009,Q009,No. While the CPI (consumer prices) and the GD...,NONE,DOC1:p27,answered
9,10,C010,Q010,"Before the crisis, European unemployment was a...",Table A-2,"DOC1:p7,DOC2:p3,DOC2:Table A-2",answered


In [6]:
if "google.colab" in sys.modules:
    files.download(str(common.OUTPUTS / "submission.csv"))